# TP1 - Classification de FashionMNIST par Transfer Learning
**Rapport associé :** `Rapport_Transfer_Learning.pdf` - ES-SAOUDY Zakaria

Ce notebook reprend les 11 parties du rapport : dataset, preprocessing ImageNet, Feature Extraction,
Fine-Tuning, entraînement, évaluation, analyse, comparaison, conclusion et améliorations.
Les valeurs du rapport sont des ordres de grandeur ; le code recalcule les résultats réellement obtenus.

## 1. Introduction
Le Transfer Learning réutilise des représentations apprises sur ImageNet afin de réduire le coût de calcul,
profiter d'extracteurs robustes (bords, textures, formes) et obtenir de bonnes performances avec moins de
données. FashionMNIST est plus difficile que MNIST et permet d'étudier le transfert entre photos RGB et
vêtements 28x28 en niveaux de gris.

## 2. Dataset FashionMNIST - 60 000 images train, 10 000 test, 10 classes

In [ ]:
%pip install -q torch torchvision scikit-learn seaborn

In [ ]:
import time, copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report, confusion_matrix

torch.manual_seed(42); np.random.seed(42)
device = torch.device('mps' if torch.backends.mps.is_available()
                      else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Appareil utilisé : {device}')
classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

## 3. Preprocessing
`Resize(224,224)` évite la disparition spatiale dans les réseaux profonds. `Grayscale(3)` duplique le canal
gris pour les poids ImageNet. La normalisation utilise exactement les moyennes et écarts-types ImageNet.

In [ ]:
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=data_transforms)
val_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=data_transforms)
batch_size = 64
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
dataloaders_dict = {'train': train_loader, 'val': val_loader}
print(f"Taille d'un batch : {next(iter(train_loader))[0].shape}")
assert next(iter(train_loader))[0].shape[1:] == (3, 224, 224)

## 4. Feature Extraction
Toutes les couches convolutionnelles sont gelées (`requires_grad=False`). Seule la tête ImageNet 1000
classes est remplacée par une couche à 10 sorties.

In [ ]:
def set_parameter_requires_grad(model):
    for param in model.parameters():
        param.requires_grad = False

def initialize_model(model_name, num_classes=10):
    if model_name == 'alexnet':
        model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
        set_parameter_requires_grad(model)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif model_name == 'vgg16':
        model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        set_parameter_requires_grad(model)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif model_name == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        set_parameter_requires_grad(model)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        raise ValueError(model_name)
    return model.to(device)

alexnet_fe = initialize_model('alexnet')
vgg16_fe = initialize_model('vgg16')
resnet18_fe = initialize_model('resnet18')

## 5. Entraînement Feature Extraction - Adam, CrossEntropyLoss, lr=1e-3, 3 epochs

In [ ]:
def train_model(model, dataloaders, criterion, optimizer, num_epochs=5):
    since = time.time(); val_acc_history = []
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}'); print('-' * 10)
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            dataloader = dataloaders[phase]
            running_loss = 0.0; running_corrects = 0
            for inputs, labels in dataloader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs); loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)
                    if phase == 'train': loss.backward(); optimizer.step()
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)
            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            if phase == 'val': val_acc_history.append(epoch_acc.item())
    elapsed = time.time() - since
    print(f'Entraînement terminé en {elapsed//60:.0f}m {elapsed%60:.0f}s')
    return model, val_acc_history

criterion = nn.CrossEntropyLoss()
fe_models = {'alexnet': alexnet_fe, 'resnet18': resnet18_fe, 'vgg16': vgg16_fe}
fe_histories = {}
for name, model in fe_models.items():
    head = model.fc if name == 'resnet18' else model.classifier[6]
    optimizer = optim.Adam(head.parameters(), lr=0.001)
    print(f'--- Feature Extraction : {name} ---')
    fe_models[name], fe_histories[name] = train_model(model, dataloaders_dict, criterion, optimizer, 3)

## 6. Fine-Tuning
Les premières couches restent gelées. Pour ResNet18, le dernier bloc `layer4` et la tête `fc` sont dégelés.
Le learning rate faible `1e-4` ajuste doucement les poids pré-entraînés.

In [ ]:
def setup_finetuning(model_name):
    model = initialize_model(model_name)
    if model_name == 'resnet18':
        for name, param in model.named_parameters():
            if 'layer4' in name or 'fc' in name:
                param.requires_grad = True
    return model

resnet18_ft = setup_finetuning('resnet18')
optimizer_ft = optim.Adam(filter(lambda p: p.requires_grad, resnet18_ft.parameters()), lr=1e-4)

## 7. Entraînement Fine-Tuning - ResNet18, 3 epochs

In [ ]:
print('--- Entraînement Fine-Tuning : ResNet18 ---')
resnet18_ft, hist_resnet_ft = train_model(
    resnet18_ft, dataloaders_dict, criterion, optimizer_ft, num_epochs=3)

## 8. Évaluation, rapport de classification et matrice de confusion

In [ ]:
def evaluate_model(model, dataloader):
    model.eval(); all_preds = []; all_labels = []
    with torch.no_grad():
        for inputs, labels in dataloader:
            outputs = model(inputs.to(device)); _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy()); all_labels.extend(labels.numpy())
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.ylabel('Vraie étiquette'); plt.xlabel('Prédiction'); plt.title('Matrice de Confusion')
    plt.tight_layout(); plt.show()
    print(classification_report(all_labels, all_preds, target_names=classes))
    return np.mean(np.asarray(all_preds) == np.asarray(all_labels))

measured_resnet_ft_accuracy = evaluate_model(resnet18_ft, val_loader)
print('Accuracy ResNet18 FT mesurée :', measured_resnet_ft_accuracy)

## 9. Analyse et interprétation
- **ResNet18 (~11M)** : les skip connections facilitent le gradient et offrent le meilleur compromis.
- **VGG16 (~138M)** : près de 90 % des paramètres sont dans les couches denses ; mémoire élevée.
- **AlexNet (~60M)** : baseline historique rapide.
- Confusions attendues : T-shirt/top vs Shirt et Pullover vs Coat, dues à leur morphologie proche.

Feature Extraction convient aux petits datasets similaires à ImageNet. Fine-Tuning convient aux datasets
plus grands ou différents, comme le passage RGB vers vêtements sur fond noir.

## 10. Comparaison finale rapportée (valeurs indicatives)
| Modèle | Paramètres | Accuracy FE | Accuracy FT | Temps | Mémoire |
|---|---:|---:|---:|---|---|
| AlexNet | ~60M | ~82% | ~88% | Très rapide | Moyenne |
| ResNet18 | ~11M | ~85% | ~93% | Rapide | Faible |
| VGG16 | ~138M | ~84% | ~91% | Lent | Élevée |

## 11. Conclusion, limites et améliorations
La duplication d'un canal gris en RGB n'est pas optimale. Le rapport propose un mécanisme d'attention
(Vision Transformer) ou un pré-entraînement sur des images en niveaux de gris. Hyperparamètres globaux :
Adam, CrossEntropyLoss, batch 64, lr FE `1e-3`, lr FT `1e-4`.